In [1]:
import pandas as pd
import requests
import json
import csv
import os
import time

In [2]:
url = "http://localhost:11434/api/generate"

modelo = "phi4-mini:3.8b"

In [3]:
provas = pd.read_csv("C:/Users/anara/Desktop/Pesquisa - Tutoria/feedback-open-source/dados - tutoria/turma_feedback_parsed.csv", sep=None, engine="python")

provas.head()

,id_atividade,disciplina,secao,tipo,aluno_id,data_resposta,email_enviado,nota_atividade,data_entrega,data_disponibilizacao,num_questao,pergunta,resposta_aluno,categoria_feedback,trecho,explicacao
0,0,Fall_24_CT-365_SE,CT (A & B),GOOGLE,ALU_643310c0e7,NaN,True,-1.0,04/10/2024,01/10/2024,1,"Does the use of standard templates, techniques...",Standard templates and techniques facilitate r...,ACERTOS,Can't guarantee,You correctly identify that using standard tem...
1,1,Fall_24_CT-365_SE,CT (A & B),GOOGLE,ALU_b3f3077d9c,NaN,True,-1.0,04/10/2024,01/10/2024,1,"Does the use of standard templates, techniques...",In refactoring we have to judge the code and p...,ACERTOS,Can't guarantee,You correctly identify that using standard tem...
2,2,Fall_24_CT-365_SE,CT (A & B),GOOGLE,ALU_a140f7b6bd,NaN,True,-1.0,04/10/2024,01/10/2024,1,"Does the use of standard templates, techniques...",Refactoring involves improving code's internal...,ACERTOS,Can't guarantee,You correctly identify that using standard tem...
3,3,Fall_24_CT-365_SE,CT (A & B),GOOGLE,ALU_51aeec6420,NaN,True,1.0,04/09/2024,01/09/2024,1,What is the significance of architectural patt...,The significance of architectural patterns in ...,ACERTOS,Organization,You correctly identify that architectural patt...
4,3,Fall_24_CT-365_SE,CT (A & B),GOOGLE,ALU_51aeec6420,NaN,True,1.0,04/09/2024,01/09/2024,1,What is the significance of architectural patt...,The significance of architectural patterns in ...,ACERTOS,Without a standard,You correctly identify that without a standard...


In [4]:
provas_sample = provas.sample(555, random_state=123).reset_index(drop=True)

print("Total de linhas na amostra:", len(provas_sample))
print("Disciplinas encontradas:")
print(provas_sample["disciplina"].unique())

Total de linhas na amostra: 555
Disciplinas encontradas:
['SUAP018548 - Arquitetura de Computadores:Bacharel em Ciência da Computação, Bacharelado, 3º Período, Turno Integral (2025)'
 'Fall_24_CT-365_SE'
 'SUAP017975 - Química Inorgânica:Técnico em Química Integrado ao Ensino Médio, Integrado, 1º Período, Turno Integral (2025)'
 '06262 - T04 - INTRODUÇÃO À INFORMÁTICA'
 '06262 - AG - INTRODUÇÃO À INFORMÁTICA' 'INTRODUÇÃO À INFORMÁTICA 2020.2'
 '[EC-2025.2] Inteligência Artificial'
 '[EC-2025.2] Introdução à Engenharia da Computação'
 'SUAP023479 - Sistemas Operacionais:Bacharel em Ciência da Computação, Bacharelado, 6º Período, Turno Integral (2025)'
 '06262 - AG - INTRODUÇÃO À INFORMÁTICA - T02'
 '06262 - CZ - INTRODUÇÃO À INFORMÁTICA - T01'
 'SUAP018004 - Química III:Técnico em Química Integrado ao Ensino Médio, Integrado, 3º Período, Turno Integral (2025)'
 'Introdução à Informática - SZ3' 'Montagem e Manutenção' 'Turma anime'
 'SUAP017859 - Química I:Técnico em Desenvolvimento de S

In [5]:
base_prompt = """
Você é um avaliador educacional.

Sua tarefa é comparar a RESPOSTA DO ALUNO com a RESPOSTA DE REFERÊNCIA e produzir um feedback específico, claro e didático sobre o desempenho apresentado.

Considere os seguintes critérios durante a avaliação:

1. Domínio da modalidade escrita formal da língua portuguesa;
2. Proximidade semântica e conceitual da resposta do aluno em relação à resposta de referência.

O feedback deve:
- indicar o que o aluno apresentou corretamente;
- identificar erros, imprecisões, omissões ou aspectos que possam ser melhorados;
- orientar de forma objetiva como a resposta poderia se aproximar mais da resposta de referência;
- considerar aspectos de clareza, coerência e correção da escrita quando forem relevantes.

IMPORTANTE:
- Compare efetivamente a resposta do aluno com a resposta de referência.
- NÃO explique ou enumere os critérios de avaliação no feedback final.
- NÃO repita as instruções desta tarefa.
- NÃO produza comentários genéricos que poderiam ser aplicados a qualquer resposta.
- Utilize elementos concretos da resposta do aluno sempre que possível.
- Não invente informações que não estejam presentes na pergunta, resposta do aluno ou resposta de referência.
- Caso a resposta esteja correta, reconheça o acerto e indique como torná-la mais completa ou precisa, quando necessário.
- Caso esteja incorreta ou incompleta, indique claramente o aspecto que deve ser corrigido ou desenvolvido.
- Produza exatamente 2 sentenças.
- Cada sentença deve conter no máximo 20 palavras.
- Escreva o feedback em português.

CONCEITO:

{concept}

----

PERGUNTA:

{question}

----

RESPOSTA DO ALUNO:

{answer_text}

----

RESPOSTA IDEAL (referência):

{reference_answer}

----

Retorne exclusivamente um objeto JSON válido no seguinte formato:

{{
    "feedback": "Texto do feedback."
}}

Não inclua análise, justificativa, explicação ou qualquer outro conteúdo fora do JSON.
"""

In [6]:
nicol_prompt = """
Você é um avaliador educacional.

Sua tarefa é comparar a RESPOSTA DO ALUNO com a RESPOSTA DE REFERÊNCIA e produzir um feedback específico, objetivo e construtivo sobre o conteúdo apresentado pelo aluno.

Construa o feedback seguindo as boas práticas propostas por Nicol & Macfarlane-Dick (2006):

1. Esclarecer metas e critérios: indicar, de forma contextualizada, o que era esperado na resposta.
2. Promover autoavaliação: mostrar onde a resposta do aluno se aproxima ou se distancia do esperado.
3. Fornecer feedback construtivo: identificar pontos fortes, limitações, erros ou aspectos incompletos.
4. Incentivar a autoestima: utilizar linguagem respeitosa e encorajadora, reconhecendo elementos corretos quando existirem.
5. Reduzir a lacuna de desempenho: indicar uma ação concreta que ajude o aluno a melhorar sua resposta.

IMPORTANTE:
- Aplique esses princípios ao conteúdo da resposta do aluno.
- NÃO explique os princípios de Nicol & Macfarlane-Dick.
- NÃO mencione no feedback final os nomes dos critérios acima.
- NÃO repita as instruções desta tarefa.
- NÃO produza perguntas genéricas sobre desempenho.
- Compare efetivamente a resposta do aluno com a resposta de referência.
- Mencione elementos concretos da resposta quando necessário.
- Não invente informações que não estejam presentes na pergunta, resposta do aluno ou resposta de referência.
- Caso a resposta esteja correta, reconheça o acerto e indique, quando pertinente, como torná-la mais completa ou precisa.
- Caso esteja incorreta ou incompleta, indique claramente o aspecto que precisa ser corrigido ou desenvolvido.
- Produza exatamente 2 sentenças.
- Cada sentença deve conter no máximo 20 palavras.
- O feedback completo deve ser escrito em português.

CONCEITO:

{concept}

----

PERGUNTA:

{question}

----

RESPOSTA DO ALUNO:

{answer_text}

----

RESPOSTA IDEAL (referência):

{reference_answer}

----

Retorne exclusivamente um objeto JSON válido no seguinte formato:

{{
    "feedback": "Texto do feedback."
}}

Não inclua qualquer texto, explicação, análise ou comentário fora do JSON.
"""

In [7]:
hattie_prompt = """
Você é um avaliador educacional.

Sua tarefa é comparar a RESPOSTA DO ALUNO com a RESPOSTA DE REFERÊNCIA
e escrever um feedback específico sobre o conteúdo produzido pelo aluno.

O feedback deve contemplar implicitamente os três componentes de Hattie & Timperley:

1. Feed-up: deixar claro o que era esperado na resposta.
2. Feedback: indicar especificamente o que o aluno acertou, errou ou deixou incompleto.
3. Feed-forward: indicar especificamente o que o aluno deve corrigir ou acrescentar.

IMPORTANTE:
- NÃO escreva as palavras "Feed-up", "Feedback" ou "Feed-forward".
- NÃO explique o modelo de Hattie & Timperley.
- NÃO repita as instruções desta tarefa.
- NÃO faça perguntas genéricas como "Onde estou indo?" ou "Como estou indo?".
- O feedback deve mencionar elementos concretos da resposta do aluno e da resposta de referência.
- Não invente informações.
- Produza exatamente 2 sentenças.
- Cada sentença deve ter no máximo 20 palavras.

DISCIPLINA:
{concept}

PERGUNTA:
{question}

RESPOSTA DO ALUNO:
{answer_text}

RESPOSTA DE REFERÊNCIA:
{reference_answer}

Retorne somente um JSON válido neste formato:

{{
    "feedback": "Texto do feedback."
}}
"""

In [13]:
nome_arquivo_csv = "phi4-mini_prova-tecnologia-results-555-base.csv"

In [14]:
schema_feedback = {
    "type": "object",
    "properties": {
        "feedback": {
            "type": "string"
        }
    },
    "required": ["explicacao"],
    "additionalProperties": False
}


cabecalhos = [
    "pergunta",
    "resposta_aluno",
    "explicacao",
    "disciplina",
    "feedback"
]

In [15]:
arquivo_existe = os.path.isfile(nome_arquivo_csv)

def gerar_feedback_phi4(prompt):
    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "format": schema_feedback,
        "options": {
            "temperature": 0.2,
            "num_predict": 300
        }
    }

    response = requests.post(
        url,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    resposta_json = response.json()
    resposta_texto = resposta_json.get("response", "")

    try:
        dados = json.loads(resposta_texto)
        return dados.get("feedback", "")

    except json.JSONDecodeError:
        print("JSON inválido retornado pelo modelo:")
        print(resposta_texto)
        return ""


with open(nome_arquivo_csv, "a", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=cabecalhos)

    if not arquivo_existe:
        escritor.writeheader()

    for i, linha in provas_sample.iterrows():
        print(f"Processando resposta {i+1}/{len(provas_sample)}")

        prompt = base_prompt.format(
            concept=linha["disciplina"],
            question=linha["pergunta"],
            answer_text=linha["resposta_aluno"],
            reference_answer=linha["explicacao"]
        )

        try:
            feedback = gerar_feedback_phi4(prompt)

        except Exception as e:
            print(f"Erro ao processar linha {i+1}: {e}")
            feedback = ""

        linha_dados = {
            "pergunta": linha["pergunta"],
            "resposta_aluno": linha["resposta_aluno"],
            "explicacao": linha["explicacao"],
            "disciplina": linha["disciplina"],
            "feedback": feedback
        }

        escritor.writerow(linha_dados)
        f.flush()

        print(f"✅ Feedback salvo ({i+1}/{len(provas_sample)})")

        
        time.sleep(1)

print("Processamento finalizado.")
print(f"Arquivo salvo em: {nome_arquivo_csv}")

Processando resposta 1/555
✅ Feedback salvo (1/555)
Processando resposta 2/555
✅ Feedback salvo (2/555)
Processando resposta 3/555
✅ Feedback salvo (3/555)
Processando resposta 4/555
✅ Feedback salvo (4/555)
Processando resposta 5/555
✅ Feedback salvo (5/555)
Processando resposta 6/555
✅ Feedback salvo (6/555)
Processando resposta 7/555
✅ Feedback salvo (7/555)
Processando resposta 8/555
✅ Feedback salvo (8/555)
Processando resposta 9/555
✅ Feedback salvo (9/555)
Processando resposta 10/555
✅ Feedback salvo (10/555)
Processando resposta 11/555
✅ Feedback salvo (11/555)
Processando resposta 12/555
✅ Feedback salvo (12/555)
Processando resposta 13/555
✅ Feedback salvo (13/555)
Processando resposta 14/555
✅ Feedback salvo (14/555)
Processando resposta 15/555
✅ Feedback salvo (15/555)
Processando resposta 16/555
✅ Feedback salvo (16/555)
Processando resposta 17/555
✅ Feedback salvo (17/555)
Processando resposta 18/555
✅ Feedback salvo (18/555)
Processando resposta 19/555
✅ Feedback salvo (